## 0 — GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/android-auto-ai-agent'
STORES=f'{DRIVE}/stores'
os.makedirs(STORES,exist_ok=True)
print('STORES =',STORES)

## 1 — Clone / update project + deps
`data/config.yaml` is edited in step 2, so it is reset before `git pull` (otherwise the
pull refuses to overwrite the local edit). Step 2 re-applies the Colab settings.

In [ ]:
import os
REPO='/content/android-auto-ai-agent'
if not os.path.isdir(REPO):
    !git clone https://github.com/appdev1307/android-auto-ai-agent.git {REPO}
else:
    !cd {REPO} && git checkout -- data/config.yaml && git pull
%cd {REPO}
!git log -1 --format='HEAD %h  %ci  %s'
!apt-get -qq install -y ripgrep >/dev/null && echo ripgrep ok
!pip -q install -r requirements.txt
print('project ready')

## 2 — Config: stores → Drive, agent → Ollama `qwen2.5-coder:32b`
Same model as `code-codegen-aosp-llm-based` (`llm_client.py`). The agent uses LangChain's
OpenAI-*compatible* client, which Ollama serves at `/v1` — no OpenAI account involved;
`OPENAI_API_KEY` is a dummy the client requires.

In [ ]:
import yaml, pathlib, os
LLM_MODEL='qwen2.5-coder:32b'
API_BASE='http://127.0.0.1:11434/v1'          # Ollama OpenAI-compatible
cfg_path=pathlib.Path('data/config.yaml'); cfg=yaml.safe_load(cfg_path.read_text())
cfg['model']['name']=LLM_MODEL
cfg['model']['api_base']=API_BASE
cfg['rag']['stores_root']=STORES
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
os.environ.pop('OPENAI_API_BASE', None)        # config is the single source of the endpoint
os.environ['OPENAI_API_KEY']='ollama'          # dummy; ignored by Ollama
print('LLM        =',LLM_MODEL,'@',API_BASE)
print('max_tokens =',cfg['model'].get('max_tokens'))
print('embed      =',cfg['rag']['embed_model'])
print('stores     =',cfg['rag']['stores_root'])

## 3 — Choose SCOPE → clone AOSP + HMI + VSS → index on GPU

`SCOPE`: `automotive` (start here) | `framework` (+frameworks/base, big) | `full` (bring your own).

Clones the **core stack** (`hardware/interfaces`, `packages/services/Car`), the **full HMI**
(Car UI Library `apps/Car/libs` + all Car apps), and **VSS** (COVESA `vehicle_signal_specification`,
dropped under `vendor/vss` so the VSS signal-tree chunker + customer priors pick it up).

Clones are shallow (`--depth 1`), tag-less, single-branch, from `AOSP_BRANCH`. Note the
store is still named `aosp15` (`AOSP_VERSION`): with `AOSP_BRANCH='main'` the tree is the
tip of AOSP main, not the `android-15.0.0_r1` release the README syncs with `repo`.
A clone that fails is skipped (non-fatal). Index runs on the free GPU (Ollama not up yet).

In [ ]:
import os, shutil, subprocess, pathlib
SCOPE='automotive'
AOSP='/content/aosp'
AOSP_BRANCH='main'        # branch on android.googlesource.com; check it exists before changing
AOSP_VERSION='aosp15'     # store name (stores/_base/<AOSP_VERSION>); keep in sync with step 5
shutil.rmtree(AOSP, ignore_errors=True)

G='https://android.googlesource.com/platform/'
REPOS={
'hardware/interfaces':           G+'hardware/interfaces',
'packages/services/Car':         G+'packages/services/Car',
'packages/apps/Car/libs':        G+'packages/apps/Car/libs',
'packages/apps/Car/Settings':    G+'packages/apps/Car/Settings',
'packages/apps/Car/SystemUI':    G+'packages/apps/Car/SystemUI',
'packages/apps/Car/Launcher':    G+'packages/apps/Car/Launcher',
'packages/apps/Car/Cluster':     G+'packages/apps/Car/Cluster',
'packages/apps/Car/Dialer':      G+'packages/apps/Car/Dialer',
'packages/apps/Car/Media':       G+'packages/apps/Car/Media',
'packages/apps/Car/Notification':G+'packages/apps/Car/Notification',
'packages/apps/Car/Radio':       G+'packages/apps/Car/Radio',
'packages/apps/Car/Messenger':   G+'packages/apps/Car/Messenger',
}

def clone(rel, url, branch=None):
    dest=f'{AOSP}/{rel}'
    os.makedirs(pathlib.Path(dest).parent, exist_ok=True)
    cmd=['git','clone','--depth','1','--no-tags','--single-branch']
    if branch:
        cmd += ['--branch', branch]
    r=subprocess.run(cmd+[url,dest], capture_output=True, text=True)
    if r.returncode==0:
        sha=subprocess.run(['git','-C',dest,'rev-parse','--short','HEAD'],
                           capture_output=True, text=True).stdout.strip()
        print(f'  cloned {rel} @ {sha}')
    else:
        print(f'  SKIP   {rel}: {r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "?"}')
    return r.returncode==0

for rel,url in REPOS.items():
    clone(rel, url, AOSP_BRANCH)

# VSS (COVESA) — default branch
clone('vendor/vss', 'https://github.com/COVESA/vehicle_signal_specification')

os.environ['AOSP_ROOT']=AOSP
print('SCOPE =',SCOPE,' branch =',AOSP_BRANCH,' store =',AOSP_VERSION)
print('--- tree ---')
!du -sh {AOSP}/hardware/interfaces {AOSP}/packages/services/Car {AOSP}/packages/apps/Car/* {AOSP}/vendor/* 2>/dev/null

### Build (or reuse) the index on Drive
The index is reused across sessions. It is rebuilt (`--reset`) when `REBUILD=True`, when
none exists, or when it was built before Update 29 (its `manifest.json` has no `git_shas`;
those stores hold absolute, pre-fix paths and stale chunks). Changing `SCOPE` also needs
`REBUILD=True`.

In [ ]:
import os, json
REBUILD=False
man_path=f'{STORES}/_base/{AOSP_VERSION}/manifest.json'
man=json.load(open(man_path)) if os.path.exists(man_path) else None
stale = man is not None and 'git_shas' not in man
scope_changed = man is not None and man.get('scope') not in (None, SCOPE)
if man is None or REBUILD or stale or scope_changed:
    why = ('no index' if man is None else 'REBUILD=True' if REBUILD else
           'pre-Update-29 index' if stale else f"scope {man.get('scope')} -> {SCOPE}")
    print('building index:', why)
    !python -m retrieval.indexer --aosp-root {AOSP} --base --aosp-version {AOSP_VERSION} --scope {SCOPE} --reset
else:
    print(f"index reused: scope={man.get('scope')} chunks={man.get('count')} "
          f"projects={len(man.get('git_shas') or {})}")
print('index at', f'{STORES}/_base/{AOSP_VERSION}')

## 4 — Start Ollama + pull model
Ollama = prebuilt, **no CUDA/torch compile**. `start_ollama()` is idempotent — re-run anytime.

The agent's system prompt (system + few-shot + skills) is ~10k tokens and `max_tokens` is
8192, so the context is 32768 — same `num_ctx` as `llm_client.py` in the codegen repo.
That repo passes `num_ctx` per request on Ollama's native `/api/generate`; the agent here
uses the OpenAI-compatible `/v1`, which does not take `num_ctx`, so it is set server-side
with `OLLAMA_CONTEXT_LENGTH` (Ollama's default 2k–4k would silently truncate the prompt).
`OLLAMA_NUM_PARALLEL=4` as in the codegen repo (fits 80 GB: ~20 GB model + 4 × ~9 GB KV).

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, time, subprocess, requests
from pathlib import Path

OLLAMA_LOG = "/content/ollama.log"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

def start_ollama():
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT,
                     env={**os.environ})
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"Ollama ready (took {i+1}s)")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama failed to start — check /content/ollama.log")

start_ollama()

In [ ]:
!/usr/local/bin/ollama pull {LLM_MODEL}
!/usr/local/bin/ollama list

### Preflight + warm-up
Loads the model once (the first agent call would otherwise pay the load time) and checks
the three things that broke earlier runs: Ollama up, config pointing at it, index present.

In [ ]:
import time, requests, yaml, os, json
cfg=yaml.safe_load(open('data/config.yaml'))
assert cfg['model']['api_base']==API_BASE and cfg['model']['name']==LLM_MODEL, 're-run step 2'
assert os.path.exists(f'{STORES}/_base/{AOSP_VERSION}/manifest.json'), 're-run step 3'
assert os.path.isdir(f'{AOSP}/packages/services/Car'), 're-run step 3 (AOSP clone)'
t=time.time()
r=requests.post(f'{API_BASE}/chat/completions', timeout=600, json={
    'model': LLM_MODEL, 'max_tokens': 8,
    'messages': [{'role': 'user', 'content': 'reply OK'}]})
r.raise_for_status()
print(f"warm-up {time.time()-t:.1f}s ->", r.json()['choices'][0]['message']['content'].strip())
!/usr/local/bin/ollama ps

## 5 — Run agent
Embedder + reranker on CPU (Ollama holds the GPU); query embedding is a few cheap calls.
Runs the full pass: seed retrieval → ReAct → commit → per-layer specialist debate →
(re-localize once if the owning specialists reject the target) → finalize.

No `--customer` flag needed: without it the agent resolves config `default_tenant`
(customer `base`) and loads `stores/_base/<AOSP_VERSION>` built in step 3.
Candidate paths are printed relative to `/content/aosp`.

A run makes roughly 10–20 LLM calls (ReAct steps, commit, up to 5 specialists × up to 3
debate rounds, patch retries, unit test), so expect minutes per bug on a 32B model.

| # | Layer | Scenario |
|---|-------|----------|
| 01 | vhal / native | speed property declared INT32 instead of FLOAT |
| 02 | carservice | DeadObjectException after CarService restart |
| 03 | startup_power | power policy not applied after STR resume |
| 04 | aidl / vhal | HVAC set returns NOT_AVAILABLE when powered off |
| 05 | native / watchdog | car watchdog kills app on health-check failure |

In [ ]:
BUG="Vehicle speed reads 0 in cluster on Android 15"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
  --logcat test_data/01_vhal_speed_type_mismatch.logcat \
  --aosp-root {AOSP} --aosp-version {AOSP_VERSION}

In [ ]:
BUG="App crashes with DeadObjectException after CarService restarts"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
  --logcat test_data/02_carservice_deadobject_restart.logcat \
  --aosp-root {AOSP} --aosp-version {AOSP_VERSION}

In [ ]:
BUG="Display power policy not applied after suspend-to-RAM resume"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
  --logcat test_data/03_power_policy_resume.logcat \
  --aosp-root {AOSP} --aosp-version {AOSP_VERSION}

In [ ]:
BUG="Setting HVAC temperature fails with NOT_AVAILABLE when HVAC power is off"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
  --logcat test_data/04_hvac_set_not_available.logcat \
  --aosp-root {AOSP} --aosp-version {AOSP_VERSION}

In [ ]:
BUG="Car watchdog kills app for unresponsive health check after boot"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
  --logcat test_data/05_carwatchdog_health_kill.logcat \
  --aosp-root {AOSP} --aosp-version {AOSP_VERSION}

## 6 — Gold labels from AOSP fix commits + eval (optional)
`eval/mine_labels.py` clones the projects again into its own cache (tag-less, shallow by
date) and keeps real bug-fix commits as labels; see README *Mining gold labels*.

Caveat for this Colab tree: the index is a `--depth 1` clone of `AOSP_BRANCH`, so fixes
merged before its tip are **already in the indexed code** (the agent sees the fixed file).
`fixed_in_index` is `null` for those (the shallow clone cannot tell). File-level
localization is still measurable; report it with that caveat, or index an older snapshot.

In [ ]:
LABELS=f'{DRIVE}/eval/labels.jsonl'
os.makedirs(f'{DRIVE}/eval', exist_ok=True)
!python -m eval.mine_labels --aosp-root {AOSP} --branch {AOSP_BRANCH} --since 2024-12-01 \
    --cache-dir /content/label_cache --mask-symbols --limit 30 \
    --out {LABELS} --review-out {DRIVE}/eval/labels_review.csv
!head -c 1500 {LABELS}

In [ ]:
!CUDA_VISIBLE_DEVICES="" python -m eval.run_eval --labels {LABELS} \
    --aosp-root {AOSP} --aosp-version {AOSP_VERSION} --k 5 --out {DRIVE}/eval/results.json

---
### Notes
- **No compile anywhere** — Ollama ships prebuilt binaries; avoids the vLLM/torch/flashinfer
  version+JIT-compile issues on Colab Python 3.13.
- **New session** → steps 0, 1, 2, 3 (AOSP clone; index reused from Drive), 4 (Ollama +
  preflight), then 5 / 6.
- **Ollama died** → re-run `start_ollama()` in step 4, then the preflight cell.
- **Change coverage** → set `SCOPE` in step 3; the index cell rebuilds when the scope differs
  (or set `REBUILD=True`).
- **Lighter model if 32B is slow/tight** → `LLM_MODEL='qwen2.5-coder:14b'` in step 2, re-run
  step 2 and the pull cell in step 4.
- **Logcat fixtures** live in `test_data/`; pass one via `--logcat`. Swap in a customer's real
  logcat the same way — but don't commit customer logs.